In [16]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    MinMaxScaler,
    OneHotEncoder,
    OrdinalEncoder,
    PolynomialFeatures,
)
from sklearn.svm import SVC


import pandas as pd

from geoai.utils_geo.RasterOps import RasterOperations
from geoai.utils_ml.ModelOps import ModelOperations

raster_ops = RasterOperations()
model_ops = ModelOperations()

In [17]:
X_train = pd.read_csv("csv_files/X_train.csv")
X_test = pd.read_csv("csv_files/X_test.csv")
y_train = pd.read_csv("csv_files/y_train.csv")
y_test = pd.read_csv("csv_files/y_test.csv")

# compute indices
X_train = raster_ops.compute_ndvi_using_df(X_train, "NIR", "RED")
X_train = raster_ops.compute_ndbi(X_train, "NIR", "SWIR")
X_train = raster_ops.compute_rei(X_train, "NIR", "BLUE")
X_test = raster_ops.compute_ndvi_using_df(X_test, "NIR", "RED")
X_test = raster_ops.compute_ndbi(X_test, "NIR", "SWIR")
X_test = raster_ops.compute_rei(X_test, "NIR", "BLUE")


# create binary and discrete NDVI
X_train = raster_ops.create_ndvi_bin(X_train, "NDVI")
X_train = raster_ops.create_ndvi_discrete(X_train, "NDVI")
X_test = raster_ops.create_ndvi_bin(X_test, "NDVI")
X_test = raster_ops.create_ndvi_discrete(X_test, "NDVI")

In [18]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression


best_params = {"C": 9.232675920286502, "max_iter": 160, "solver": "lbfgs"}
lr = LogisticRegression(**best_params)
bag = BaggingClassifier(estimator=lr,
                        n_estimators=500,
                        max_samples=100,
                        max_features=3,
                        bootstrap=True,
                        bootstrap_features=False,
                        n_jobs=-1,
                        random_state=1)


# create a pipeline
pipeline = model_ops.make_pipeline(bag)
print(pipeline)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('poly',
                                                                   PolynomialFeatures())]),
                                                  ['BLUE', 'GREEN', 'RED',
                                                   'NIR', 'SWIR', 'NDVI',
                                                   'NDBI', 'REI']),
                                                 ('onehot',
                                                  OneHotEncoder(dtype=<class 'int'>),
                                                  ['NDVI_bin']),
                                                 ('ordinal',
                                                  OrdinalEncoder(categories=[['low_veg',
                                                                              'medium_veg',
                                                                         

In [19]:
# Fit the VotingClassifier
pipeline.fit(X_train, y_train)

# Predict the labels of the test set
y_train_pred = pipeline.predict(X_train)
y_test_pred = pipeline.predict(X_test)

# Calculate the accuracy of the VotingClassifier
print(f"Train Accuracy: {model_ops.calculate_classification_accuracy(y_train, y_train_pred)}")
print(f"Train Accuracy: {model_ops.calculate_classification_accuracy(y_test, y_test_pred)}")

d:\Projects\GEOAI\GeoAI-ISPRS-SS\geoai-env\Lib\site-packages\sklearn\utils\validation.py:1339: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
d:\Projects\GEOAI\GeoAI-ISPRS-SS\geoai-env\Lib\site-packages\sklearn\ensemble\_bagging.py:888: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


Train Accuracy: (0.938501291989664, 0.9388048000267161, 0.938501291989664, 0.9386097334635456)
Train Accuracy: (0.9276859504132231, 0.9288275892386143, 0.9276859504132231, 0.9279844457530407)
